# Week 1 prompt baselines (v8)
Runs B0/B1/B2 (context conditions) and S1/S2/S3 (prompting strategies) on v8 dev split only, excluding holdout. Uses existing loaders/runner/scoring and leakage protections (dev-only, exclude eval row, exclude holdout, exclude same-document/ambiguous identities; runner holdout-prompt assert). `RUN=False` by default to avoid live endpoint calls.

In [23]:
# Parameters
ARMS = ["B0", "B1", "B2", "S1", "S2", "S3"]  # B0-B2 context conditions; S1-S3 prompting strategies
OUT_DIR = "results/week1_prompt_baseline_v8/runs"
RUN = False  # set True to execute live endpoint calls (only when legitimately available)


In [24]:
import os, sys, json
from pathlib import Path
_root = Path.cwd().resolve()
if not (_root / 'src').is_dir():
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
from src.config import GOLD_PATH
from src.eval.gold_rows import load_rows
import importlib
import src.eval.prompting.inputs as _inputs
importlib.reload(_inputs)
from src.eval.prompting.inputs import load_dev_rows, load_holdout_ids
import src.eval.contexts as _contexts
importlib.reload(_contexts)
from src.eval.contexts import load_few_shot as load_dev_few_shot
import src.eval.prompting.fewshot as _pfew
importlib.reload(_pfew)
from src.eval.prompting.fewshot import load_few_shot as load_s1_few_shot
from src.eval.prompting.runner import run_arm
all_rows = load_rows(GOLD_PATH, active_only=True) or [json.loads(l) for l in open(GOLD_PATH) if l.strip()]
dev_rows = load_dev_rows()
holdout_ids = load_holdout_ids()
print('v8 gold:', GOLD_PATH)
print('total active:', len(all_rows), 'dev:', len(dev_rows), 'holdout_excluded:', len(holdout_ids))
assert holdout_ids and not any(r.get('id') in holdout_ids for r in dev_rows)
sample = dev_rows[0]
sdocs = {sample.get('document_id')} | {e.get('document_id') for e in (sample.get('evidence') or []) if isinstance(e,dict)}
sdocs = {d for d in sdocs if isinstance(d,str) and d}
b0_fs = load_dev_few_shot(n=3, exclude_id=sample['id'], exclude_doc_ids=sdocs)
s1_fs = load_s1_few_shot()
print('leakage_checks: holdout excluded from dev =', True)
print('leakage_checks: B0 dev few-shot excludes eval row/same-doc =', sample['id'] not in [x['id'] for x in b0_fs] and all(not ({x.get('document_id')} & sdocs) for x in b0_fs))
print('leakage_checks: S1 static few-shot count (leakage-filtered) =', len(s1_fs))
print('arms_configured:', ARMS)
manifests = []
if RUN:
    for arm in ARMS:
        manifests.append(run_arm(arm, rows=dev_rows, smoke=False, live=True, out_dir=OUT_DIR))
    print('runs_written:', len(manifests))
else:
    print('RUN=False: no live endpoint calls executed.')


v8 gold: data/eval/datasets/gold_v2_eval_v8_public.jsonl
total active: 97 dev: 73 holdout_excluded: 24
leakage_checks: holdout excluded from dev = True
leakage_checks: B0 dev few-shot excludes eval row/same-doc = True
leakage_checks: S1 static few-shot count (leakage-filtered) = 3
arms_configured: ['B0', 'B1', 'B2', 'S1', 'S2', 'S3']
RUN=False: no live endpoint calls executed.


**Experiment semantics (preserved):**
- **Context conditions:** B0 closed-book, B1 oracle context, B2 retrieved/RAG context.
- **Prompting strategies:** S1 structured output + few-shot, S2 CoT/self-consistency (private reasoning not exposed in scored outputs), S3 decomposition (extract→compute→answer).
- Dev split only; holdout fully excluded; leakage protections enforced by loaders/runner. Metrics reported from stored run records when available.

In [25]:
import json
from pathlib import Path
from collections import defaultdict
from src.eval.prompting.inputs import load_dev_rows
dev_by_id = {r['id']: r for r in load_dev_rows()}
base = Path(OUT_DIR)
mfiles = sorted(base.glob('*.manifest.json')) if base.exists() else []
def is_completed(m):
    if m.get('stopped_reason'): return False
    n = int(m.get('n_processed_this_run') or 0)
    return n >= len(dev_by_id)
def metric(r, k):
    s = r.get('score') or {}
    v = s.get(k)
    return 1 if v is True else (0 if v is False else None)
def mean(vals):
    xs=[v for v in vals if v is not None]
    return (sum(xs)/len(xs)) if xs else None
completed, incomplete = [], []
for mp in mfiles:
    m = json.loads(mp.read_text(encoding='utf-8'))
    rp = Path(m.get('output_path',''))
    if not rp.is_absolute(): rp = Path.cwd()/rp
    m['_rp']=rp
    (completed if is_completed(m) else incomplete).append(m)
print('dev_rows:', len(dev_by_id), 'manifests:', len(mfiles), 'completed:', len(completed), 'incomplete/excluded:', len(incomplete))
for m in incomplete:
    print('INCOMPLETE/EXCLUDED (not a completed baseline):', m.get('arm'), m.get('run_id'), 'processed=', m.get('n_processed_this_run'), 'stopped_reason=', m.get('stopped_reason'), m.get('output_path'))
if not completed:
    print('\nPENDING: no completed baseline runs for B0,B1,B2,S1,S2,S3; accuracy/citation/abstention/latency/cost metrics unavailable (not invented). Existing B0 probe, if present, is excluded above.')
for m in completed:
    recs=[json.loads(l) for l in m['_rp'].read_text(encoding='utf-8').splitlines() if l.strip()]
    ans=[r for r in recs if dev_by_id.get(r.get('row_id'),{}).get('expected_behaviour')=='answer']
    by_sub, by_doc = defaultdict(list), defaultdict(list)
    for r in recs:
        row=dev_by_id.get(r.get('row_id'),{})
        sub=row.get('abstention_subtype') if row.get('expected_behaviour')=='abstain' else 'answer'
        by_sub[sub].append(r); by_doc[row.get('doc_type')].append(r)
    def agg(rs):
        lat=[r.get('latency_ms') for r in rs if isinstance(r.get('latency_ms'), (int,float))]
        pt=sum(((r.get('usage') or {}).get('prompt_tokens') or 0) for r in rs)
        ct=sum(((r.get('usage') or {}).get('completion_tokens') or 0) for r in rs)
        return {'n':len(rs), 'abstain_correct':mean([metric(r,'abstain_correct') for r in rs]),
                'false_answer':mean([metric(r,'false_answer') for r in rs]), 'false_abstention':mean([metric(r,'false_abstention') for r in rs]),
                'citation_correct_answerable':mean([metric(r,'citation_valid') for r in rs if dev_by_id.get(r.get('row_id'),{}).get('expected_behaviour')=='answer']),
                'numeric_match_applicable':mean([metric(r,'numeric_match') for r in rs if (r.get('score') or {}).get('numeric_match') is not None]),
                'mean_latency_ms': (sum(lat)/len(lat) if lat else None), 'prompt_tokens':pt, 'completion_tokens':ct}
    overall_acc = mean([metric(r,'numeric_match') if (r.get('score') or {}).get('numeric_match') is not None else metric(r,'entity_match') for r in ans])
    print('\nARM', m.get('arm'), '| kind:', ('context condition' if m.get('arm') in ('B0','B1','B2') else 'prompting strategy'), '| run', m.get('run_id'))
    print('overall:', {**agg(recs), 'answer_accuracy_applicable': overall_acc})
    print('abstention_breakdown_by_subtype:', {k: agg(v) for k,v in sorted(by_sub.items(), key=lambda x:str(x[0]))})
    print('abstention_breakdown_by_doc_type_annual_quarterly:', {k: agg(v) for k,v in sorted(by_doc.items()) if k in ('annual','quarterly')})


dev_rows: 73 manifests: 1 completed: 0 incomplete/excluded: 1
INCOMPLETE/EXCLUDED (not a completed baseline): B0 20261119T122358Z_B0 processed= 1 stopped_reason= error rate 1/1 exceeded 5% results/week1_prompt_baseline_v8/runs/20261119T122358Z_B0_B0.jsonl

PENDING: no completed baseline runs for B0,B1,B2,S1,S2,S3; accuracy/citation/abstention/latency/cost metrics unavailable (not invented). Existing B0 probe, if present, is excluded above.


In [ ]:
# Holdout (24 rows) reserved for final report only; excluded from all dev experiments/runs.
